In [ ]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Multiply, BatchNormalization, Dropout
from tensorflow.keras.models import Model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelBinarizer

df = pd.read_csv('../data/processed/parking_data.csv')

lb = LabelBinarizer()
weather_encoded = lb.fit_transform(df['weather'])
df['weather_0'] = weather_encoded[:, 0]
df['weather_1'] = weather_encoded[:, 1]
df['weather_2'] = weather_encoded[:, 2]

df_sample = df.sample(2000, random_state=42).reset_index(drop=True)


train_df, val_df = train_test_split(df_sample, test_size=0.2, random_state=42)

print(f"Training samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")

Training samples: 1600
Validation samples: 400


In [ ]:

def process_data(filepath, weather_vec, label):

    img = tf.io.read_file(filepath)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [150, 150])
    img = img / 255.0
    
    return (img, weather_vec), label

def create_dataset(dataframe, batch_size=32):
    filepaths = dataframe['filepath'].values
    weather_data = dataframe[['weather_0', 'weather_1', 'weather_2']].values.astype('float32')
    labels = dataframe['label'].values.astype('float32')
    
    dataset = tf.data.Dataset.from_tensor_slices((filepaths, weather_data, labels))
    dataset = dataset.map(process_data, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return dataset

BATCH_SIZE = 32
train_dataset = create_dataset(train_df, BATCH_SIZE)
val_dataset = create_dataset(val_df, BATCH_SIZE)

print("Data Pipeline Created Successfully!")

Data Pipeline Created Successfully!


In [ ]:

image_input = Input(shape=(150, 150, 3), name='image_input')

x = Conv2D(32, (3, 3), activation='relu')(image_input)
x = MaxPooling2D((2, 2))(x)

x = Conv2D(64, (3, 3), activation='relu')(x)
x = MaxPooling2D((2, 2))(x)
x = BatchNormalization()(x) 

x = Conv2D(128, (3, 3), activation='relu')(x)
x = MaxPooling2D((2, 2))(x)

x = Flatten()(x)
image_features = Dense(64, activation='relu')(x)

weather_input = Input(shape=(3,), name='weather_input')

w = Dense(16, activation='relu')(weather_input)
weather_gate = Dense(64, activation='sigmoid', name='weather_gate')(w)

gated_features = Multiply(name='Condition_Gating')([image_features, weather_gate])

out = Dense(32, activation='relu')(gated_features)
out = Dropout(0.5)(out) 
final_output = Dense(1, activation='sigmoid', name='occupancy_output')(out)

model = Model(inputs=[image_input, weather_input], outputs=final_output)

model.compile(optimizer='adam', 
              loss='binary_crossentropy', 
              metrics=['accuracy'])

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image_input         │ (None, 150, 150,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 148, 148,  │        896 │ image_input[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 74, 74,    │          0 │ conv2d[0][0]      │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 72, 72,    │     18,496 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 36, 36,    │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 36, 36,    │        256 │ max_pooling2d_1[… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 34, 34,    │     73,856 │ batch_normalizat… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 17, 17,    │          0 │ conv2d_2[0][0]    │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ weather_input       │ (None, 3)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 36992)     │          0 │ max_pooling2d_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 16)        │         64 │ weather_input[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 64)        │  2,367,552 │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ weather_gate        │ (None, 64)        │      1,088 │ dense_1[0][0]     │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Condition_Gating    │ (None, 64)        │          0 │ dense[0][0],      │
│ (Multiply)          │                   │            │ weather_gate[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 32)        │      2,080 │ Condition_Gating… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 32)        │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ occupancy_output    │ (None, 1)         │         33 │ dropout[0][0]     │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,464,321 (9.40 MB)

 Trainable params: 2,464,193 (9.40 MB)

 Non-trainable params: 128 (512.00 B)

In [ ]:
print("Starting training on a small subset to verify the architecture...")

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=5  
)

model.save('../models/smart_parking_test_model.h5')
print("Test model saved successfully!")

Starting training on a small subset to verify the architecture...
Epoch 1/5
50/50 ━━━━━━━━━━━━━━━━━━━━ 21s 360ms/step - accuracy: 0.8006 - loss: 0.4425 - val_accuracy: 0.4875 - val_loss: 0.9199
Epoch 2/5
50/50 ━━━━━━━━━━━━━━━━━━━━ 17s 344ms/step - accuracy: 0.9081 - loss: 0.2467 - val_accuracy: 0.4875 - val_loss: 1.5713
Epoch 3/5
50/50 ━━━━━━━━━━━━━━━━━━━━ 21s 410ms/step - accuracy: 0.9206 - loss: 0.2269 - val_accuracy: 0.4875 - val_loss: 1.0272
Epoch 4/5
50/50 ━━━━━━━━━━━━━━━━━━━━ 20s 399ms/step - accuracy: 0.9331 - loss: 0.1768 - val_accuracy: 0.4900 - val_loss: 1.4622
Epoch 5/5
50/50 ━━━━━━━━━━━━━━━━━━━━ 20s 396ms/step - accuracy: 0.9425 - loss: 0.1495 - val_accuracy: 0.4900 - val_loss: 1.5466


Test model saved successfully!
